In [0]:
trigger_location = dbutils.widgets.get("trigger_location")
prompt = dbutils.widgets.get("prompt")
frame_stride = int(dbutils.widgets.get("frame_stride"))
truncate = dbutils.widgets.get("truncate")
threshold = float(dbutils.widgets.get("threshold"))

# trigger_location = '/Volumes/pubsec_video_processing/cv/auto_segment/images/maren_jack.MOV'
# prompt = 'girl in pink sweatshrit'
# frame_stride = 30
# truncate = True
# threshold = 0.2

if truncate==0 or truncate=='false' or truncate=='False':
    truncate = False
else:
    truncate=True

print(trigger_location)
print(prompt)
print(frame_stride)
print(truncate)
print(type(truncate))
print(threshold)
print(type(threshold))

/Volumes/pubsec_video_processing/cv/auto_segment/images/maren_jack.MOV
girl in pink sweatshrit
30
True
<class 'bool'>
0.2
<class 'float'>


In [0]:
# dbutils.notebook.exit("Notebook execution stopped by user request.")

In [0]:
import mlflow
import imageio
import numpy as np
import pandas as pd
import cv2
import base64
import timeit
from io import BytesIO
from PIL import Image
import os

In [0]:
from datetime import datetime

if trigger_location.endswith('/') and (trigger_location[-4]!='.' or trigger_location[-5]!='.'):
    # Your volume directory path
    directory_path = trigger_location

    # List all files in the directory
    files = dbutils.fs.ls(directory_path)

    # Filter out directories, keep only files
    files = [f for f in files if not f.isDir()]

    # Sort by modification time (most recent first)
    files_sorted = sorted(files, key=lambda x: x.modificationTime, reverse=True)

    # Get the most recent file
    if files_sorted:
            most_recent_file = files_sorted[0]
            most_recent_path = most_recent_file.path.replace('dbfs:','')
            most_recent_name = most_recent_file.name
            
            print(f"Most recent file: {most_recent_name}")
            print(f"Full path: {most_recent_path}")
            print(f"Modified: {datetime.fromtimestamp(most_recent_file.modificationTime/1000)}")
    else:
            print("No files found in directory")
else:
    most_recent_file = dbutils.fs.ls(trigger_location)[0]
    most_recent_path = most_recent_file.path.replace('dbfs:','')
    most_recent_name = most_recent_file.name

    print(f"Most recent file: {most_recent_name}")
    print(f"Full path: {most_recent_path}")
    print(f"Modified: {datetime.fromtimestamp(most_recent_file.modificationTime/1000)}")

Most recent file: maren_jack.MOV
Full path: /Volumes/pubsec_video_processing/cv/auto_segment/images/maren_jack.MOV
Modified: 2026-04-07 18:13:41


In [0]:
# most_recent_file = trigger_location
# most_recent_name = most_recent_file.split("/")[-1]
# most_recent_path = most_recent_file.replace('dbfs:','')

In [0]:
from huggingface_hub import login
import os

hf_pat = dbutils.secrets.get("justinm-buildathon-secrets", "hf_pat")
os.environ["HF_TOKEN"] = hf_pat
login(token=hf_pat)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [0]:
import mlflow

model = mlflow.pyfunc.load_model("models:/pubsec_video_processing.cv.transformers-sam3-video@job")

/databricks/python/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Unexpected internal error when monkey patching `Trainer.train`: cannot import name 'HybridCache' from 'transformers' (/local_disk0/.ephemeral_nfs/envs/pythonEnv-2e89d496-aad6-404f-919a-e25b35b8340b/lib/python3.12/site-packages/transformers/__init__.py)
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda
loading model...
loading processor...
context loaded


In [0]:
def write_results(FILE_URL, results, truncate=True, frame_stride=30, speed_up=5):
    import os

    if FILE_URL.startswith("/Volumes/pubsec_video_processing/cv/auto_segment/inputs/"):
        OUTPUT_FILE_URL = FILE_URL.replace("inputs", "outputs")
        output_dir = os.path.dirname(OUTPUT_FILE_URL)
    else:
        OUTPUT_FILE_URL = most_recent_path.replace(most_recent_name, f"outputs/{most_recent_name}")
        output_dir = os.path.dirname(OUTPUT_FILE_URL)
    os.makedirs(output_dir, exist_ok=True)
    FPS = 30

    def decode_mask(encoded_mask: str) -> np.ndarray:
        """Decode base64 mask back to numpy array"""
        buf = BytesIO(base64.b64decode(encoded_mask))
        return np.load(buf)

    def add_text_overlays(frame: np.ndarray, timestamp_sec: float, count_text: str) -> np.ndarray:
        """Add timestamp overlay to frame in the top-right corner"""
        # Convert seconds to HH:MM:SS.ms format
        hours = int(timestamp_sec // 3600)
        minutes = int((timestamp_sec % 3600) // 60)
        seconds = int(timestamp_sec % 60)
        milliseconds = int((timestamp_sec % 1) * 1000)
            
        timestamp_text = f"{hours:02d}:{minutes:02d}:{seconds:02d}.{milliseconds:03d}"
        
        # Create a copy to avoid modifying original
        frame_with_overlay = frame.copy()
        
        # Get frame dimensions
        height, width = frame.shape[:2]
        
        # Set up text properties
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.5
        font_thickness = 2
        text_color = (255, 255, 255)  # White text
        bg_color = (0, 0, 0)  # Black background
        padding = 10
        
        # Get text size
        (text_width_right, text_height_right), baseline_right = cv2.getTextSize(
            timestamp_text, font, font_scale, font_thickness
        )
        
        # Position in top-right corner
        text_x_right = width - text_width_right - padding
        text_y_right = padding + text_height_right
        
        # Draw semi-transparent background rectangle
        bg_x1_right = text_x_right - 5
        bg_y1_right = text_y_right - text_height_right - 5
        bg_x2_right = text_x_right + text_width_right + 5
        bg_y2_right = text_y_right + baseline_right + 5

        # Get text size
        (text_width_left, text_height_left), baseline_left = cv2.getTextSize(
            count_text, font, font_scale, font_thickness
        )

        # Position in top-left corner
        text_x_left = padding
        text_y_left = padding + text_height_left
        
        # Draw semi-transparent background rectangle
        bg_x1_left = text_x_left - 5
        bg_y1_left = text_y_left - text_height_left - 5
        bg_x2_left = text_x_left + text_width_left + 5
        bg_y2_left = text_y_left + baseline_left + 5

        # Create overlay for semi-transparency
        overlay = frame_with_overlay.copy()
        cv2.rectangle(overlay, (bg_x1_right, bg_y1_right), (bg_x2_right, bg_y2_right), bg_color, -1)
        cv2.rectangle(overlay, (bg_x1_left, bg_y1_left), (bg_x2_left, bg_y2_left), bg_color, -1)
        cv2.addWeighted(overlay, 0.6, frame_with_overlay, 0.4, 0, frame_with_overlay)
        
        # Draw text
        cv2.putText(
            frame_with_overlay,
            timestamp_text,
            (text_x_right, text_y_right),
            font,
            font_scale,
            text_color,
            font_thickness,
            cv2.LINE_AA
        )
        # Draw text
        cv2.putText(
            frame_with_overlay,
            count_text,
            (text_x_left, text_y_left),
            font,
            font_scale,
            text_color,
            font_thickness,
            cv2.LINE_AA
        )
        return frame_with_overlay

    # Detect input video format and setup codec
    import os
    _, input_ext = os.path.splitext(FILE_URL)
    _, output_ext = os.path.splitext(OUTPUT_FILE_URL)
    
    # Use the same extension as input if output doesn't have one
    if not output_ext:
        OUTPUT_FILE_URL = OUTPUT_FILE_URL + input_ext
        output_ext = input_ext
    
    print(f"Input format: {input_ext}, Output format: {output_ext}")
    
    # Open original video to get frames
    print("Processing frames and applying masks...")
    cap = cv2.VideoCapture(FILE_URL)
    fps = cap.get(cv2.CAP_PROP_FPS) or FPS
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Video properties: {frame_width}x{frame_height} @ {fps} fps")

    # Create a mapping of frame_idx to results
    print(f"Found {len(results)} frames in results")
    result_map = {r["frame_idx"]: r for r in results}
    # print("keys", result_map.keys())

    # Initialize imageio writer for direct frame-by-frame writing with H.264
    import imageio
    import tempfile
    with tempfile.NamedTemporaryFile(suffix=output_ext, delete=False) as tmp_file:
        temp_video_path = tmp_file.name
    
    output_fps = (fps/frame_stride) * speed_up if frame_stride is not None else fps * speed_up
    
    # Use imageio's get_writer for frame-by-frame writing with H.264 (via bundled ffmpeg)
    # This automatically uses imageio-ffmpeg which bundles ffmpeg binaries
    video_writer = imageio.get_writer(
        temp_video_path,
        fps=output_fps,
        codec='libx264',
        pixelformat='yuv420p',
        quality=8  # Quality: 0 (worst) to 10 (best)
    )
    
    print(f"Writing video at {output_fps:.2f} fps using imageio with H.264 codec...")
    
    frame_idx = 0
    saved_images = []
    frames_written = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Only process frames with segmentation results
        if frame_idx in result_map:
            # print(frame_idx)
            # Convert BGR to RGB
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            res = result_map[frame_idx]

            if res["masks"]:
                # Get all of the masks
                masks = [decode_mask(x) for x in res["masks"]]
                no_of_masks = len(masks)
                
                # Overlay with transparency
                overlay = rgb_frame.copy()
                for mask in masks:
                    overlay[mask > 0.5] = [0, 255, 0]  # Green overlay
                masked_frame = cv2.addWeighted(rgb_frame, 0.7, overlay, 0.3, 0)
                                
            else:
                masked_frame = rgb_frame
                no_of_masks = 0
                        
            # Calculate timestamp for this frame
            timestamp_sec = frame_idx / fps
            
            # Add timestamp overlay (convert back to BGR for cv2 operations, then back to RGB)
            masked_frame_bgr = cv2.cvtColor(masked_frame, cv2.COLOR_RGB2BGR)
            # masked_frame_with_timestamp = add_timestamp(masked_frame_bgr, timestamp_sec)
            # masked_frame_with_timestamp_and_count = add_text_overlay(masked_frame_with_timestamp, f"Count: {len(res['masks'])}")
            masked_frame_with_text_overlays = add_text_overlays(masked_frame_bgr, timestamp_sec, f"Count: {no_of_masks}")
            masked_frame = cv2.cvtColor(masked_frame_with_text_overlays, cv2.COLOR_BGR2RGB)
            
            # Save original frame to saved_images
            if not truncate:
                saved_images.append(Image.fromarray(rgb_frame))
                # Write segmented frame directly to video using imageio (expects RGB)
                video_writer.append_data(masked_frame)
                frames_written += 1
            elif res["masks"]:
                saved_images.append(Image.fromarray(rgb_frame))
                # Write segmented frame directly to video using imageio (expects RGB)
                video_writer.append_data(masked_frame)
                frames_written += 1
            
            # if frames_written % 100 == 0:
            #     print(f"Written {frames_written} frames")
            
        frame_idx += 1

    cap.release()
    video_writer.close()  # Close imageio writer
    
    print(f"Saved {len(saved_images)} frames to memory")
    print(f"Written {frames_written} frames to video with H.264 encoding")

    # 3. Copy video to final destination
    import shutil
    
    temp_size = os.path.getsize(temp_video_path)
    print(f"Video created with H.264: {temp_size:,} bytes ({temp_size/1024/1024:.2f} MB)")

    # Copy to final destination
    print(f"Copying to Volumes: {OUTPUT_FILE_URL}")
    shutil.copy2(temp_video_path, OUTPUT_FILE_URL)

    final_size = os.path.getsize(OUTPUT_FILE_URL)
    print(f"✓ Video successfully saved to: {OUTPUT_FILE_URL}")
    print(f"  Final size: {final_size:,} bytes ({final_size/1024/1024:.2f} MB)")

    # Clean up temporary file
    if os.path.exists(temp_video_path):
        os.remove(temp_video_path)
        print("Cleaned up temporary file")

    return saved_images

In [0]:
def process_file(triggered_file, prompt, resize=True):
    print(f"Triggered by file: {triggered_file}")

    model_input = pd.DataFrame([{
        "video_path": triggered_file,
        "prompt": prompt,
        "frame_stride": frame_stride,  # Process every nth frame
        "batch_size": 32,
        "threshold": threshold,
        "mask_threshold": 0.5
    }])
    results = model.predict(model_input)
    return results

In [0]:
print("Files processed in this run:", most_recent_path)

print("Segmenting video...")
starting_time = timeit.default_timer()
results = process_file(most_recent_path, prompt, resize=False)
print(f"Inference time: {round((timeit.default_timer() - starting_time))} secs")

Files processed in this run: /Volumes/pubsec_video_processing/cv/auto_segment/images/maren_jack.MOV
Segmenting video...
Triggered by file: /Volumes/pubsec_video_processing/cv/auto_segment/images/maren_jack.MOV
/Volumes/pubsec_video_processing/cv/auto_segment/images/maren_jack.MOV
girl in pink sweatshrit
30
4
0.2
Inference time: 33 secs


In [ ]:
len(results)

In [0]:
print("Writing segmented video...")
starting_time = timeit.default_timer()
written_images = write_results(most_recent_path, results, truncate=truncate, frame_stride=frame_stride, speed_up=5)
print(f"Writing time: {round((timeit.default_timer() - starting_time))} secs")

Processing frames and applying masks...
Found 33 frames in results


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1920, 1080) to (1920, 1088) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Saved 33 frames to memory
Saved 33 segmented frames to memory
Writing video to temporary file...
Temporary video created: 2,747,322 bytes (2.62 MB)
Copying to Volumes: /Volumes/pubsec_video_processing/cv/auto_segment/images/outputs/maren_jack.MOV
✓ Video successfully saved to: /Volumes/pubsec_video_processing/cv/auto_segment/images/outputs/maren_jack.MOV
  Final size: 2,747,322 bytes (2.62 MB)
Cleaned up temporary file


In [0]:
import io
import numpy
import base64
import cv2
import asyncio
import numpy as np
from openai import AsyncOpenAI, ChatCompletion
from typing import NamedTuple, List, Optional, Generator, Tuple

# helpers
class FrameBatch(NamedTuple):
    content: List[dict]
    sizes: List[int]
    total_bytes: int
    
    @property
    def frame_count(self) -> int:
        return len(self.content)

def encode_jpeg(frame: np.ndarray, quality: int) -> bytes:
    """cv2.imencode is 3-5x faster than PIL, no color conversion needed."""
    _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return buf.tobytes()

def to_content(jpg_bytes: bytes) -> dict:
    b64 = base64.b64encode(jpg_bytes).decode('ascii')
    return {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}}

def fit_to_size(frame: np.ndarray, max_size: int) -> Tuple[bytes, int]:
    
    # Early exit - most frames fit at q=95
    data = encode_jpeg(frame, 95)
    if len(data) <= max_size:
        return data, 95
    
    # binary search FTW!
    lo, hi, best = 10, 94, (encode_jpeg(frame, 10), 10)
    while lo <= hi:
        mid = (lo + hi) >> 1
        data = encode_jpeg(frame, mid)
        if len(data) <= max_size:
            best, lo = (data, mid), mid + 1
        else:
            hi = mid - 1
    return best

def process_frame(
    frame: np.ndarray, 
    quality: Optional[int] = None,
    max_size: int = 500_000
) -> Tuple[dict, int, int]:
    if quality:
        data = encode_jpeg(frame, quality)
    else:
        data, quality = fit_to_size(frame, max_size)
    return to_content(data), quality, len(data)

def stream_content(
    video: cv2.VideoCapture,
    quality: Optional[int] = None,
    max_size: int = 500_000
) -> Generator[Tuple[dict, int, int], None, None]:
    while True:
        ok, frame = video.read()
        if not ok:
            return
        yield process_frame(frame, quality, max_size)

def batch_content(
    video: cv2.VideoCapture,
    quality: Optional[int] = None,
    max_frame_size: int = 500_000,
    max_batch_size: int = 3_000_000
) -> Generator[FrameBatch, None, None]:
    max_frame_size = min(max_frame_size, max_batch_size)
    batch, sizes, batch_size = [], [], 0
    for content, _, size in stream_content(video, quality, max_frame_size):
        if batch and batch_size + size > max_batch_size:
            yield FrameBatch(batch, sizes, batch_size)
            batch, sizes, batch_size = [], [], 0
        batch.append(content)
        sizes.append(size)
        batch_size += size
    
    if batch:
        yield FrameBatch(batch, sizes, batch_size)

def stream_content_from_images(
    images: List[Image.Image],
    quality: Optional[int] = None,
    max_size: int = 500_000
) -> Generator[Tuple[dict, int, int], None, None]:
    for img in images:
        # Convert PIL Image to numpy array (OpenCV format)
        frame = np.array(img)
        # Convert RGB to BGR if needed (OpenCV uses BGR)
        if len(frame.shape) == 3 and frame.shape[2] == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        yield process_frame(frame, quality, max_size)

def batch_content_from_images(
    images: List[Image.Image],
    quality: Optional[int] = None,
    max_frame_size: int = 500_000,
    max_batch_size: int = 3_000_000
) -> Generator[FrameBatch, None, None]:
    max_frame_size = min(max_frame_size, max_batch_size)
    batch, sizes, batch_size = [], [], 0
    for content, _, size in stream_content_from_images(images, quality, max_frame_size):
        if batch and batch_size + size > max_batch_size:
            yield FrameBatch(batch, sizes, batch_size)
            batch, sizes, batch_size = [], [], 0
        batch.append(content)
        sizes.append(size)
        batch_size += size
    
    if batch:
        yield FrameBatch(batch, sizes, batch_size)

def summarize_frames(frame_batch: FrameBatch):
    return oai.chat.completions.create(
    model=FMAPI_MODEL,
    messages=[
        {'role': 'user', 'content': [
            {'type': 'text', 'text': 
                'Describe what is happening in the following sequence of images '
                'which represent a frames from a video. Describe what is going on in from the cameras '
                'perspective. Be descriptive - but succinct and functional'
            },
            *frame_batch.content
        ]}
    ]
)
    
def get_content(completion: ChatCompletion) -> str:
    '''
        get_content extracts the textual content.  Required with gemini 3 models
        as g3 model's content may (or may not) include a thoughtSignature which 
        is a reference to the model's reasoning process.
    '''
    content = completion.choices[0].message.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return content[0]['text']
    else:
        print(type(content))
        print(content)
        raise Exception('weird content signature')

In [0]:
# actual running code
# vid_path = output_file_url
# video = cv2.VideoCapture(vid_path)
# frame_batches = list(batch_content(video, quality=80))
# video.release()

frame_batches = list(batch_content_from_images(written_images, quality=80))

FMAPI_SERVING_URL = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}'
FMAPI_BASE_URL = f'{FMAPI_SERVING_URL}/serving-endpoints'
print(f'🎯 FMAPI_BASE_URL: {FMAPI_BASE_URL}')
FMAPI_API_TOKEN = dbutils.secrets.get('justinm-buildathon-secrets', 'db_pat')
FMAPI_MODEL = 'stsu-gemini-3-flash' 
# FMAPI_MODEL = 'databricks-gemini-2-5-flash'

# clients
oai = AsyncOpenAI(
    api_key = FMAPI_API_TOKEN,
    base_url = FMAPI_BASE_URL
)

# Get results
ai_results = await asyncio.gather(*[
    summarize_frames(frame_batch)
    for frame_batch in frame_batches
])

# concatenation of previous generations
context = []
for idx, result in enumerate(ai_results):
    description = f'SECTION {idx}\n' + get_content(result)
    context.append(description)

context = '\n'.join(context)

video_summary = await oai.chat.completions.create(
    model=FMAPI_MODEL,
    messages=[
        {'role': 'user', 'content': [
            {'type': 'text', 'text': 
                'Summarize the following sections of a video with a focus on '
                'what is functionally happening over time. Be descriptive. '
                'The resulting summary should be a functional narrative. '
                'Return this as Markdown text - and only markdown. '
            },
            {'type': 'text', 'text': context}
        ]}
    ]
)

text = get_content(video_summary)
print(text)

🎯 FMAPI_BASE_URL: https://fe-vm-industry-solutions-buildathon.cloud.databricks.com/serving-endpoints
The video begins with two children, a girl and a boy, engaged in cookie activities at a table. The girl is initially seen decorating a rectangular cookie with sprinkles, holding it up briefly, and then placing it back down. She then selects a round cookie from a tray and hands it to the boy, who begins to examine it. Concurrently, the girl starts arranging other rectangular cookies upright on the parchment paper.

The scene then shifts to show both children actively preparing cookies for baking. The girl takes pieces of dough from a large slab and rolls them into small, round balls. Throughout this process, she frequently looks up and appears to be speaking, sometimes towards the boy and sometimes directly at the camera. Simultaneously, the boy consistently focuses on his own piece of dough, holding it in his cupped hands and diligently shaping it into a ball. A juice box is visible nex

In [0]:
txt_filename = most_recent_path.split('.')[0] + '.txt'
txt_filename = txt_filename.replace('/inputs/', '/descriptions/')
print(txt_filename)

dbutils.fs.put(txt_filename, text, True)

/Volumes/pubsec_video_processing/cv/auto_segment/images/maren_jack.txt
Wrote 1253 bytes.


True